In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [2]:
block_size = 8  # n-1 gram
embedding_size = 16
hidden_size = 512
batch_size = 512
steps = 20000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(
    dimension=0, size=block_size, step=1
)
y_train = torch.tensor([ctoi[ch] for ch in train[block_size:]], dtype=torch.long)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(
    dimension=0, size=block_size, step=1
)
y_val = torch.tensor([ctoi[ch] for ch in val[block_size:]], dtype=torch.long)

In [4]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, vocab_size, embedding_size, block_size, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size),
            nn.Flatten(),
            nn.Linear(
                in_features=embedding_size * block_size, out_features=hidden_size
            ),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=vocab_size),
        )

    def forward(self, x):
        return self.layers(x)

In [5]:
# model initialization
model = MultiLayerPerceptron(
    vocab_size=vocab_size,
    embedding_size=embedding_size,
    block_size=block_size,
    hidden_size=hidden_size,
)
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    train_logits = model(x_train[batch])
    train_loss = nn.functional.cross_entropy(input=train_logits, target=y_train[batch])

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits, target=y_val)
            print(
                f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}"
            )

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.150618076324463, val loss = 4.142927646636963


Step 1000: train loss = 1.6847399473190308, val loss = 1.8927888870239258


Step 2000: train loss = 1.582754373550415, val loss = 1.8345553874969482


Step 3000: train loss = 1.6898882389068604, val loss = 1.8015073537826538


Step 4000: train loss = 1.597341537475586, val loss = 1.786741018295288


Step 5000: train loss = 1.5563347339630127, val loss = 1.7732412815093994


... (11 outputs left out)

Step 17000: train loss = 1.4467542171478271, val loss = 1.7399014234542847


Step 18000: train loss = 1.4057681560516357, val loss = 1.742966651916504


Step 19000: train loss = 1.5202877521514893, val loss = 1.7517951726913452


Step 19999: train loss = 1.3595128059387207, val loss = 1.7537356615066528


In [7]:
# prediction
with torch.inference_mode():
    output = ["\n"] * block_size
    max_output_length = 1000
    while len(output) < max_output_length:
        i = torch.tensor([[ctoi[ch] for ch in output[-block_size:]]], dtype=torch.long)

        logits = model(i)
        P = torch.softmax(input=logits, dim=1)

        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()

        output.append(itoc[j])

In [8]:
print("".join(output[block_size:]))

KING RICHARD II:
What examples' -tweel what wast serves, this lady hervessed thank draw you, for fearful daughter:
But from this?
Why royalord, thum iunt
And Shis drawern, boyster.

LEONTES:
It, sweet matter.

First Citizen:
Within this soble at thy dasher here's that
What say offens me flank. Hencused by the sun
to all to priest ope
poite is, satill's the lord o' the prince,
Thine it non his fliately:
Clowere pawlaupen sopening blother's told thousay
Is thine
Or rokes-u bucking how I clock tell her so return attes,
That I bOse:
I meet anmish all thy hours yay nothing,
Or Captaffordon empelare the reggar, though hat when have pression, make to't,
Let's herucior home, his what I, all acreent be and sputhfuls stower, chamblic sit off;
TunguaM suray dazes buttle alone, and graven and quakes within mowardone that lamest down or owans,
So Dugge our stone by your king; power:
And make thee them go apply.

Second Murderer:
Why ragged the rose of the garlands, sincerous of a presently 
